In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from config import SAMPLE_NUM, DISEASES, MISSING_DATA_LEVELS, IMPUTATION_METHODS, RANDOM_STATE, MISS_TYPES, get_random_num

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.calibration import LabelEncoder
from sklearn.base import clone

rng_num = get_random_num()

scaler = StandardScaler()

classification_models = {
    "logisticRegression": LogisticRegression(penalty='l2'),
    "randomForest": RandomForestClassifier(n_estimators=100, random_state=rng_num),
    "svm": SVC(probability=True, random_state=rng_num),
    "xgboost": XGBClassifier(n_estimators=100, learning_rate=0.1, eval_metric='logloss', random_state=rng_num)
}

TRAIN_DATA_PATH = "imputed_train_splits"
TEST_DATA_PATH = "missing_test_splits"
OUTPUT_FOLDER = Path("classification_results")
OUTPUT_FOLDER.mkdir(exist_ok=True)

def load_train_test_samples(train_folder_path, test_folder_path, disease_name, miss_type, miss_prcnt, imp_method):
    miss_prcnt = int(miss_prcnt * 100)
    df_samples = []
    for i in range(SAMPLE_NUM):
        train_folder = Path(train_folder_path + f"/sample{i+1}")
        train_data_file = train_folder / f"{disease_name}_train_{miss_type}{miss_prcnt}_{imp_method}.csv"
        test_folder = Path(test_folder_path + f"/sample{i+1}")
        test_data_file = test_folder / f"{disease_name}_test_miss{miss_prcnt}.csv"

        if train_data_file.exists() and test_data_file.exists():
            train_df = pd.read_csv(train_data_file)
            test_df = pd.read_csv(test_data_file)
            df_samples.append((train_df, test_df))
        else:
            print(f"Warning: Missing files for: sample {i+1}, disease: {disease_name}, missing percentage: {miss_prcnt}, imputation method: {imp_method}. Skipping these files.")
    
    return df_samples

def load_original_data(folder_path):
    folder = Path(folder_path)
    df_list = []
    file_list = []
    for file in folder.glob("*.csv"):
        df_list.append(pd.read_csv(file))
        file_list.append(file)
    return file_list, df_list

In [8]:
def process_categorical_features(X_train, X_test):
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns
    if len(cat_cols) > 0:
        X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True, dtype=float)
        X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True, dtype=float)
        
        X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
        
    return X_train, X_test

def train_on_imputed_data(name, train_df, test_df, model_name, model):
    # last col is target
    target_col = train_df.columns[-1]

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col].astype(int)
    
    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col].astype(int)

    X_train, X_test = process_categorical_features(X_train, X_test)

    le = LabelEncoder()
    le.fit(pd.concat([y_train, y_test]))
    y_train = le.transform(y_train)
    y_test = le.transform(y_test)

    if len(np.unique(y_train)) < 2:
        print(f"Target in {name} does not have at least 2 classes - skipping.")
        return None, None, None, None

    model = clone(model)
    if (model_name in ["svm", "logisticRegression"]):
        X_train_model = scaler.fit_transform(X_train)
        X_test_model = scaler.transform(X_test)
    else:
        X_train_model = X_train.values
        X_test_model = X_test.values
    model.fit(X_train_model, y_train)

    y_pred = model.predict(X_test_model)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test_model)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test_model)
    else:
        y_score = None

    acc = accuracy_score(y_test, y_pred)

    classes_in_train = np.unique(y_train)
    try:    
        if len(classes_in_train) == 2:
            f1 = f1_score(y_test, y_pred, zero_division=0)
            if y_score is not None:
                if y_score.ndim == 1:
                    auc = roc_auc_score(y_test, y_score)
                else:
                    auc = roc_auc_score(y_test, y_score[:, 1])
            else:
                auc = np.nan
        else:
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
            if y_score is not None:
                try:
                    auc = roc_auc_score(y_test, y_score, multi_class='ovr', labels=classes_in_train)
                except ValueError as e:
                    print(f"Error calculating AUC for {name} with {model_name}: {e}")
                    auc = np.nan
            else:
                auc = np.nan
    except ValueError as e:
        print(f"Error calculating AUC for {name} with {model_name}: {e}")
        auc = np.nan

    report = classification_report(y_test, y_pred, zero_division=0)

    return acc, f1, auc, report

def train_on_full_data(name, df, model_name, model):
    target_col = df.columns[-1]

    X = df.drop(columns=[target_col])
    y = df[target_col].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    X_train, X_test = process_categorical_features(X_train, X_test)

    le = LabelEncoder()
    le.fit(pd.concat([y_train, y_test]))
    y_train = le.transform(y_train)
    y_test = le.transform(y_test)

    if len(np.unique(y_train)) < 2:
        print(f"Target in {name} does not have at least 2 classes - skipping.")
        return None, None, None, None

    model = clone(model)
    if (model_name in ["svm", "logisticRegression"]):
        X_train_model = scaler.fit_transform(X_train)
        X_test_model = scaler.transform(X_test)
    else:
        X_train_model = X_train.values
        X_test_model = X_test.values
    model.fit(X_train_model, y_train)

    y_pred = model.predict(X_test_model)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test_model)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test_model)
    else:
        y_score = None

    acc = accuracy_score(y_test, y_pred)

    classes_in_train = np.unique(y_train)
    try:    
        if len(classes_in_train) == 2:
            f1 = f1_score(y_test, y_pred, zero_division=0)
            if y_score is not None:
                if y_score.ndim == 1:
                    auc = roc_auc_score(y_test, y_score)
                else:
                    auc = roc_auc_score(y_test, y_score[:, 1])
            else:
                auc = np.nan
        else:
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
            if y_score is not None:
                try:
                    auc = roc_auc_score(y_test, y_score, multi_class='ovr', labels=classes_in_train)
                except ValueError as e:
                    print(f"Error calculating AUC for {name} with {model_name}: {e}")
                    auc = np.nan
            else:
                auc = np.nan
    except ValueError as e:
        print(f"Error calculating AUC for {name} with {model_name}: {e}")
        auc = np.nan

    report = classification_report(y_test, y_pred, zero_division=0)

    return acc, f1, auc, report

***Train models on imputed data***

In [18]:
for disease in DISEASES:
    for miss_prcnt in MISSING_DATA_LEVELS:
        miss_prcnt_int = int(miss_prcnt * 100)
        for imp_method in IMPUTATION_METHODS:
            for miss_type in MISS_TYPES:
                train_test_pairs = load_train_test_samples(TRAIN_DATA_PATH, TEST_DATA_PATH, disease, miss_type, miss_prcnt, imp_method)
                for model_name, model in classification_models.items():
                    full_name = f"{disease}_{miss_type}{miss_prcnt_int}_{imp_method}"
                    acc_all = []
                    f1_all = []
                    auc_all = []
                    for i, (train_df, test_df) in enumerate(train_test_pairs):
                        acc, f1, auc, report = train_on_imputed_data(full_name, train_df, test_df, model_name, model)
                        acc_all.append(acc)
                        f1_all.append(f1)
                        auc_all.append(auc)
                        out_report_folder = Path(f"classification_reports/sample{i+1}")
                        out_report_folder.mkdir(exist_ok=True, parents=True)
                        out_report_file = out_report_folder / f"{full_name}_{model_name}.txt"
                        with open(out_report_file, "w", encoding="utf-8") as f:
                            f.write(report)
                        print(f"Full report saved to: {out_report_file}")
                    avg_acc = np.mean(acc_all)
                    sd_acc = np.std(acc_all)
                    avg_f1 = np.mean(f1_all)
                    sd_f1 = np.std(f1_all)
                    avg_auc = np.mean(auc_all)
                    sd_auc = np.std(auc_all)
                    result_file = OUTPUT_FOLDER / f"{full_name}_{model_name}.txt"
                    with open(result_file, "w", encoding="utf-8") as f:
                        f.write(f"Accuracy: {avg_acc:.14f}\n")
                        f.write(f"Accuracy std: {sd_acc:.14f}\n")
                        f.write(f"F1 Score: {avg_f1:.14f}\n")
                        f.write(f"F1 Score std: {sd_f1:.14f}\n")
                        f.write(f"AUC: {avg_auc:.14f}\n")
                        f.write(f"AUC std: {sd_auc:.14f}\n")
                    print(f"Average results saved to: {result_file}")

Full report saved to: classification_reports\sample1\hepatitis_mar10_mean_logisticRegression.txt
Full report saved to: classification_reports\sample2\hepatitis_mar10_mean_logisticRegression.txt
Full report saved to: classification_reports\sample3\hepatitis_mar10_mean_logisticRegression.txt
Full report saved to: classification_reports\sample4\hepatitis_mar10_mean_logisticRegression.txt
Full report saved to: classification_reports\sample5\hepatitis_mar10_mean_logisticRegression.txt
Average results saved to: classification_results\hepatitis_mar10_mean_logisticRegression.txt
Full report saved to: classification_reports\sample1\hepatitis_mar10_mean_randomForest.txt
Full report saved to: classification_reports\sample2\hepatitis_mar10_mean_randomForest.txt
Full report saved to: classification_reports\sample3\hepatitis_mar10_mean_randomForest.txt
Full report saved to: classification_reports\sample4\hepatitis_mar10_mean_randomForest.txt
Full report saved to: classification_reports\sample5\hepat

***Train models on original data***

In [ ]:
ORIGINAL_PATH = "processed_datasets"

file_list, df_list = load_original_data(ORIGINAL_PATH)

for path, df in zip(file_list, df_list):
    name = path.stem
    if name != "diabetes":
        continue
    for model_name, model in classification_models.items():
        acc, f1, auc, report = train_on_full_data(name, df, model_name, model)
        out_report_folder = Path(f"classification_reports")
        out_report_folder.mkdir(exist_ok=True, parents=True)
        out_report_file = out_report_folder / f"{name}_none00_none_{model_name}.txt"
        with open(out_report_file, "w", encoding="utf-8") as f:
            f.write(report)
        print(f"Full report saved to: {out_report_file}")
        result_file = OUTPUT_FOLDER / f"{name}_none00_none_{model_name}.txt"
        with open(result_file, "w", encoding="utf-8") as f:
            f.write(f"Accuracy: {acc:.14f}\n")
            f.write(f"Accuracy std: 0.00000000000000\n")
            f.write(f"F1 Score: {f1:.14f}\n")
            f.write(f"F1 Score std: 0.00000000000000\n")
            f.write(f"AUC: {auc:.14f}\n")
            f.write(f"AUC std: 0.00000000000000\n")
        print(f"Results saved to: {result_file}")

Full report saved to: classification_reports\diabetes_none00_none_logisticRegression.txt
Results saved to: classification_results\diabetes_none00_none_logisticRegression.txt
Full report saved to: classification_reports\diabetes_none00_none_randomForest.txt
Results saved to: classification_results\diabetes_none00_none_randomForest.txt
Full report saved to: classification_reports\diabetes_none00_none_svm.txt
Results saved to: classification_results\diabetes_none00_none_svm.txt
Full report saved to: classification_reports\diabetes_none00_none_xgboost.txt
Results saved to: classification_results\diabetes_none00_none_xgboost.txt
